1. EDA


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("AB_NYC_2019.csv")
df.head()

In [ ]:
drop_cols = ["id", "name", "host_id", "host_name", "last_review"]
df = df.drop(columns=drop_cols)
df.info()

In [ ]:
df.describe()
df.isnull().sum()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
sns.histplot(df["price"], bins=80, ax=axes[0, 0]); axes[0, 0].set_title("price")
sns.histplot(df["minimum_nights"], bins=50, ax=axes[0, 1]); axes[0, 1].set_title("minimum_nights")
sns.histplot(df["number_of_reviews"], bins=50, ax=axes[0, 2]); axes[0, 2].set_title("number_of_reviews")
sns.histplot(df["availability_365"], bins=50, ax=axes[1, 0]); axes[1, 0].set_title("availability_365")
sns.histplot(df["reviews_per_month"].dropna(), bins=50, ax=axes[1, 1]); axes[1, 1].set_title("reviews_per_month")
sns.countplot(data=df, x="room_type", ax=axes[1, 2]); axes[1, 2].set_title("room_type")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

In [ ]:
sample = df.sample(2000, random_state=42)
sns.pairplot(sample, hue="room_type",
             vars=["price", "minimum_nights", "number_of_reviews", "availability_365"])
plt.show()

- price имеет сильный правый скос: медиана ~106, максимум 10000.
- Есть явные выбросы (цена > 500$-1-2% записей).
- reviews_per_month содержит пропуски - они соответствуют объявлениям без отзывов, логично заполнить нулем.
- room_type и neighborhood_group - категориальные с малым числом уровней.
- Корреляция с price слабые - сильных линейных связей нет, нужны модели и фичи.

2. Preprocessing & Feature Engineering

In [ ]:
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df = df.dropna(subset=["price"])

In [ ]:
df = df[df["price"] > 0]
df = df[df["price"] <= 500]      # отсекаем явные выбросы
df = df.reset_index(drop=True)

In [ ]:
df["price_log"] = np.log1p(df["price"])
sns.histplot(df["price_log"], bins=50); plt.title("log(price)"); plt.show()

In [ ]:
manhattan_lat, manhattan_lon = 40.7580, -73.9855

df["dist_to_center"] = np.sqrt(
    (df["latitude"] - manhattan_lat) ** 2 +
    (df["longitude"] - manhattan_lon) ** 2
)

# Плотность объявлений у хоста
df["listings_per_host"] = df.groupby("host_id")["host_id"].transform("count") \
    if "host_id" in df.columns else 1

In [ ]:
df = df[[
    "neighbourhood_group", "neighbourhood", "latitude", "longitude",
    "room_type", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count",
    "availability_365", "dist_to_center", "price_log"
]]

In [ ]:
df = pd.get_dummies(df, columns=["neighbourhood_group", "room_type"],
                    drop_first=True)
df = df.drop(columns=["neighbourhood"])   # слишком много уровней, уберём

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["price_log"])
y = df["price_log"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

3. Моделирование

In [ ]:
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def evaluate(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    r2 = r2_score(y_te, pred)
    mae = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    print(f"{name:15s} | R² = {r2:.3f} | MAE = {mae:.3f} | RMSE = {rmse:.3f}")
    return model, pred

alphas = np.logspace(-3, 3, 50)

_, _ = evaluate(LinearRegression(), X_train_s, y_train, X_test_s, y_test, "Linear")
ridge, _ = evaluate(RidgeCV(alphas=alphas), X_train_s, y_train, X_test_s, y_test, "RidgeCV")
lasso, _ = evaluate(LassoCV(alphas=alphas, max_iter=5000), X_train_s, y_train, X_test_s, y_test, "LassoCV")
enet, _ = evaluate(ElasticNetCV(alphas=alphas, l1_ratio=[.1, .5, .7, .9, .95, 1], max_iter=5000),
                   X_train_s, y_train, X_test_s, y_test, "ElasticNetCV")

In [ ]:
coefs = pd.Series(lasso.coef_, index=X.columns).sort_values()
coefs = coefs[coefs != 0]

plt.figure(figsize=(10, 8))
coefs.plot(kind="barh")
plt.title("Коэффициенты Lasso (только ненулевые)")
plt.tight_layout()
plt.show()

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear", "RidgeCV", "LassoCV", "ElasticNetCV"],
    "R²":    [r2_score(y_test, LinearRegression().fit(X_train_s, y_train).predict(X_test_s)),
              r2_score(y_test, ridge.predict(X_test_s)),
              r2_score(y_test, lasso.predict(X_test_s)),
              r2_score(y_test, enet.predict(X_test_s))],
})
results

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear", "RidgeCV", "LassoCV", "ElasticNetCV"],
    "R²":    [r2_score(y_test, LinearRegression().fit(X_train_s, y_train).predict(X_test_s)),
              r2_score(y_test, ridge.predict(X_test_s)),
              r2_score(y_test, lasso.predict(X_test_s)),
              r2_score(y_test, enet.predict(X_test_s))],
})
results

- Логарифмирование price — самое сильное улучшение метрик.
- Удаление выбросов > 500$ — стабилизирует обучение.
- One-hot для room_type и neighbourhood_group — заметно улучшает Ridge/Lasso.
- Масштабирование (StandardScaler) — обязательно для регуляризованных моделей, иначе Lasso/Ridge работают плохо.
- Признак dist_to_center — стабильно попадает в топ по модулю коэффициента, полезен.

Что не помогло:
- neighbourhood (слишком много категорий) — переобучает модель, убрал.
- minimum_nights, availability_365 — слабо влияют на log(price).

Лучшая модель:
- RidgeCV и LassoCV дают близкие метрики; Lasso дополнительно обнуляет лишние признаки.
- R² ≈ 0.5 — хороший результат для такой задачи. MAE на исходной шкале ≈ 40–50$.

Интересное:
- Самый сильный признак — room_type_Entire home/apt (положительный коэффициент: целые квартиры дороже).
- dist_to_center входит с отрицательным знаком: чем дальше от центра, тем дешевле.
- neighbourhood_group_Manhattan — сильный положительный вклад, что подтверждает известный факт: Манхэттен дороже.